# 02 — Baseline Pruner Evaluation Results

Run all baseline pruning strategies and compare results.

In [ ]:
# Mount Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or update repo
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Mrtuzy/CENG467_Final.git"
REPO_DIR = Path("/content/CENG467_Final")

if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    print(f"Removing non-git directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Working directory: {Path.cwd()}")


In [ ]:
# Minimal dependencies for baseline sanity checks
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "datasets",
    "rank_bm25",
    "tqdm",
    "pandas",
    "matplotlib",
    "scikit-learn",
])


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from src.evaluation.runner import EvaluationRunner
from src.utils.io import load_jsonl

In [ ]:
# Load data
from pathlib import Path
import subprocess

DATA_PATH = Path("data/processed/hotpotqa_experiments.jsonl")
if not DATA_PATH.exists():
    subprocess.check_call([sys.executable, "src/data/splitter.py"])

samples = load_jsonl(str(DATA_PATH))[:20]
print(f"Loaded {len(samples)} samples for evaluation")


In [ ]:
# Run all baseline pruners with dry_run mode
pruners = ["no_pruning", "naive_truncation", "recomp"]
results_by_pruner = {}

for pruner in pruners:
    print(f"\nEvaluating {pruner}...")
    output_path = f"experiments/results/{pruner}_results.jsonl"
    runner = EvaluationRunner(
        pruner_name=pruner,
        generator_model="dry_run",
        judge_model="dry_run",
        output_path=output_path,
    )
    agg = runner.run(samples)
    results_by_pruner[pruner] = agg
    print(f"  Faithfulness: {agg['faithfulness_mean']:.3f}")
    print(f"  Compression: {agg['compression_ratio_mean']:.3f}")

In [ ]:
# Create comparison table
df = pd.DataFrame([
    {
        'Pruner': name,
        'Faithfulness': agg['faithfulness_mean'],
        'EM': agg['em_mean'],
        'F1': agg['f1_mean'],
        'Compression': agg['compression_ratio_mean'],
        'Latency (s)': agg['latency_mean_s'],
    }
    for name, agg in results_by_pruner.items()
])

print("\n" + "="*80)
print(df.to_string(index=False))
print("="*80)

In [ ]:
# Plot: Faithfulness vs Compression
plt.figure(figsize=(8, 6))
plt.scatter(
    df['Compression'],
    df['Faithfulness'],
    s=200,
    alpha=0.7,
    c=range(len(df)),
    cmap='viridis'
)
for i, row in df.iterrows():
    plt.annotate(row['Pruner'], (row['Compression'], row['Faithfulness']),
                xytext=(5, 5), textcoords='offset points')

plt.xlabel('Compression Ratio')
plt.ylabel('Faithfulness Score')
plt.title('Pruning Strategies: Faithfulness vs Compression')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('experiments/results/faithfulness_vs_compression.png', dpi=150)
plt.show()

In [ ]:
# Save outputs to Drive (optional)
import os
import shutil

drive_root = "/content/drive/MyDrive/CENG467_Final"
src_dir = "experiments/results"
dst_dir = os.path.join(drive_root, "experiments", "results")

if os.path.exists("/content/drive"):
    if os.path.exists(src_dir):
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f"Copied {src_dir}/ to {dst_dir}")
    else:
        print(f"Source folder not found: {src_dir}")
else:
    print("Drive not mounted. Run the Drive mount cell first.")